### Methods for reading HDF5 files and exporting to dictionary structure.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import os
import re

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Threshold for peak-to-average ratio acceptance of image.
THRESHOLD = 10


def h5_to_dict(filename):
    """Read an HDF5 file into a nested dict.

    Structure:
        data[group_name]['attrs']        -> group attributes (metadata)
        data[group_name][dataset_name]   -> dict with 'data' (numpy array)
                                           and 'attrs' (dataset attributes)
    """
    def _read_group(grp):
        out = {'attrs': dict(grp.attrs)}
        for name, item in grp.items():
            if isinstance(item, h5py.Dataset):
                out[name] = {'data': item[()], 'attrs': dict(item.attrs)}
            elif isinstance(item, h5py.Group):
                out[name] = _read_group(item)
        return out

    with h5py.File(filename, 'r') as f:
        return {key: _read_group(f[key]) for key in f}


def data_from_h5_files(files):
    """Read multiple HDF5 files into a nested dict."""
    data = {}
    for filename in files:
        key = re.sub('pass', '', os.path.basename(filename).split('_')[2].split('-')[0])
        data[key] = h5_to_dict(filename)
    return data


def files_in_directory(wdir, pattern):
    """List files in a directory matching a regex pattern."""
    rawfiles = os.listdir(wdir)
    return sorted([f"{wdir}/{f}" for f in rawfiles if re.match(pattern, f)])


def beam_visible(img, cx, cy, droi=4, threshold=THRESHOLD):
    """Determine if a beam is present in the image based on the peak-to-average ratio."""
    roi_avg = np.mean(img[cy-droi:cy+droi, cx-droi:cx+droi])
    mean = np.mean(img)
    ratio_rtom = roi_avg / mean if mean != 0 else 0
    
    return ratio_rtom >= threshold


def beam_properties(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return properties of the beam profiles from the data dict.
    
    Args:
        dataset (dict): Nested dict containing the set of one full scan (one pass).

    Returns:
        beam_propties : A dict mapping scan numbers to (cx, cy), (fx, fy) centroids and FWHMs.
        beam_images   : A dict mapping scan numbers to DVF images of the beam, only scans where
                        the beam is visible are returned
    """
    beam_propties = {}
    beam_images   = {}
    xval          = []
    for scan, scandata in dataset.items():
        # Extract scanning index and observable value.
        sc = int(scan.split('-')[-1])
        xval = float(scandata['attrs'].get(dev_motor)[0])

        # Get image data and calculate centroid.
        img = scandata['dvf_B1']['data']

        # Centroids.
        cx = np.sum(img, axis=0).argmax()
        cy = np.sum(img, axis=1).argmax()

        # FWHMs
        x_profile = img[cy, :]
        y_profile = img[:, cx]

        fwhm_x = np.sum(x_profile > (x_profile.max() / 2))
        fwhm_y = np.sum(y_profile > (y_profile.max() / 2))
        
        # Do not register properties if there is no beam image.
        if not beam_visible(img, cx, cy, droi, threshold):
            continue

        # Dict is indexed by scan number.
        beam_propties[sc] = [xval, [cx, cy], [fwhm_x, fwhm_y]]
        beam_images[sc]   = img
 
    return beam_images, beam_propties


def beam_centroid(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return centroids of the beam profiles from the data dict.
    
    Args:
        data (dict): Nested dict containing the set of one full scan (one pass).

            Returns:
        dict: A dict mapping scan numbers to (cx, cy) centroids.
    """
    centroids = {}
    xval      = []
    
    _, beam_propties = beam_properties(dataset, dev_motor, droi, threshold)

    for sc in beam_propties.keys():
        xval   = beam_propties[sc][0]
        cx, cy = beam_propties[sc][1]

        centroids[sc] = [xval, [cx, cy]]
 
    return centroids


def beam_fwhm(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return fwhm of the beam profiles from the data dict.
    
    Args:
        data (dict): Nested dict containing the set of one full scan (one pass).

            Returns:
        dict: A dict mapping scan numbers to (fx, fy) fwhm's.
    """
    fwhms = {}
    _, beam_propties = beam_properties(dataset, dev_motor, droi, threshold)

    for sc in beam_propties.keys():
        xval   = beam_propties[sc][0]
        fx, fy = beam_propties[sc][2]
        
        fwhms[sc] = [xval, [fx, fy]]
 
    return fwhms


def beam_intensity(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return the total intensity of the beam profiles from the data dict.
    
    Args:
        data (dict): Nested dict containing the set of one full scan (one pass).

            Returns:
        dict: A dict mapping scan numbers to total intensity.
    """

    beam_images, beam_propties = beam_properties(dataset, dev_motor, 
                                                 droi, threshold)
    exptime = dataset['scan-0000']['dvf_B1']['attrs']['expo_time']

    intensities = {}
    droi = 2
    for sc in beam_images.keys():
        img            = beam_images[sc]
        xval           = beam_propties[sc][0]
        cx, cy         = beam_propties[sc][1]
        fwhm_x, fwhm_y = beam_propties[sc][2]

        peak = np.mean(img[cy - droi : cy + droi + 1,
                           cx - droi : cx + droi + 1])

        mask = img > (peak / 2)
        area_mask  = np.sum(mask)
        img_masked = np.where(mask, img, 0)
        area_img_masked = np.sum(img_masked)

        intensity_by_mask = (area_img_masked / (area_mask * exptime)
                             if area_mask != 0 else 0)

        peak /= exptime
        peak_fwhm_norm = (peak / (fwhm_x * fwhm_y)
                          if fwhm_x * fwhm_y != 0 else 0)
        
        intensities[sc] = [xval, [peak, intensity_by_mask, peak_fwhm_norm]]
    
    return intensities


def observable_data(dataset, observable):
    """Extract the behavior of a variable across scans in a dataset.
    
    Args:
        dataset (dict): Nested dict containing the set of
            one full scan (one pass).
        observable (str): The variable to extract
            (e.g., 'photocollector', 'centroid').

    Returns:
        tuple: (motor, scans, xval, yval) where:
            motor (str)      : The name of the scanned variable.
            scans (np.array) : Array of scan numbers.
            xval (np.array)  : Array of scanned variable values.
            yval (list of np.array): List of arrays containing the observable
                values for each scan.
    """
    # The scanned variable (idx).
    motor     = dataset['scan-0000']['attrs']['scan_motor']
    device    = dataset['scan-0000']['attrs']['scan_type']
    dev_motor = f"{device}.{motor}"

    # Centroids are calculated in beam_centroid().
    if observable == 'centroid':
        centroids = beam_centroid(dataset, dev_motor)

        # Dict is ordered by scan number.
        scans = np.array(list(centroids.keys()))

        # Observable values.
        xvals = np.array([centroids[sc][0] for sc in scans])

        # Cetroid values.
        values = np.array([centroids[sc][1] for sc in scans])
        centr  = [values[:, 0], values[:, 1]]

        return motor, scans, xvals, centr

    # FWHMs are calculated in beam_fwhm().
    if observable == 'fwhm':
        fwhms = beam_fwhm(dataset, dev_motor)

        # Dict is ordered by scan number.
        scans = np.array(list(fwhms.keys()))

        # Observable values.
        xvals = np.array([fwhms[sc][0] for sc in scans])

        # Cetroid values.
        values = np.array([fwhms[sc][1] for sc in scans])
        fs     = [values[:, 0], values[:, 1]]

        return motor, scans, xvals, fs

    # Intensities are calculated in beam_intensity().
    if observable == 'intensity':
        intensities = beam_intensity(dataset, dev_motor)

        # Dict is ordered by scan number.
        scans = np.array(list(intensities.keys()))

        # Observable values.
        xvals = np.array([intensities[sc][0] for sc in scans])

        # Cetroid values.
        values      = np.array([intensities[sc][1] for sc in scans])
        intensities = [values[:, 0], values[:, 1], values[:, 2]]

        return motor, scans, xvals, intensities

    scans, xval, yval = [], [], []
    # Run over each point scanned.
    for scan, scandata in dataset.items():
        # Get scan number, scanning index and observable value.
        scans.append(int(scan.split('-')[-1]))
        xmeta = scandata['attrs'].get(dev_motor)
        ymeta = scandata['attrs'].get(f"{device}.{observable}")

        # Append the values, handling both scalar and array metadata cases.
        if isinstance(xmeta, list) or isinstance(xmeta, np.ndarray):
            xval.append(float(xmeta[0]))
        else:
            xval.append(float(xmeta))
        if isinstance(ymeta, list) or isinstance(ymeta, np.ndarray):
            yval.append(float(ymeta[0]))
        else:
            yval.append(float(ymeta))

    return motor, np.array(scans), np.array(xval), [np.array(yval)]


def scan_plot(data, observable, first_item=0, last_item=None,
              gr=1, labels=None):
    """Plot the behavior of an observable across scans for each dataset.
    
    Args:
        data (dict): Nested dict containing the set of all passes.
        observable (str): The variable to plot (e.g., 'photocollector', 'centroid').
        first_item (int): The first point of the scan to be included in the plot.
            It allows skipping initial points if desired. Default is 0 (include all points).
        last_item (int or None): The last point of the scan to be included in the plot.
            If None, include all points.
        gr (int): The number of subplots to create. Default is 1.
        labels (list of str or None): Custom labels for each subplot. Default is None.
    """

    if gr == 1:
        fig, ax = plt.subplots(figsize=(15, 6))
        ax = [ax]
    else:
        fig, ax = plt.subplots(nrows=1, ncols=gr, figsize=(20, 6))

    # Loop over each dataset and plot the observable vs. scan number.
    # Each dataset corresponds to a different pass.
    for datakey, dataset in data.items():
        motor, scans, xval, yvals = observable_data(dataset, observable)

        for idx, yval in enumerate(yvals):
            if last_item is None:
                ax[idx].plot(xval[first_item:], yval[first_item:],
                        marker='o', label=f'Dataset {datakey}')
            else:
                ax[idx].plot(xval[first_item:last_item],
                             yval[first_item:last_item],
                             marker='o', label=f'Dataset {datakey}')

    # Graph appearance settings.
    for idx in range(gr):
        ax[idx].set_xlabel(motor.capitalize())
        ax[idx].set_ylabel(f"{observable.capitalize()}"
                           f" {labels[idx] if labels else ''}")
        ax[idx].legend()
        ax[idx].grid()

    plt.show()


def centroid_plot(dataset, scanpass, wdir='.', save_fmt='gif'):
    """Plot the beam profile images and their centroids in a dataset.
    
    Args:
        dataset (dict): Nested dict containing the set of one full scan
            (one pass).
        scanpass (str): Identifier for the scan pass
            (used in titles and filenames).
        wdir (str): Working directory to save the plots.
            Default is current directory.
        save_fmt (str): Format to save the animation
            ('gif', 'mp4', or '' to skip saving).
    """
    # Use the existing function to get motor, positions and centroids.
    (motor, scan_nums, xval,
     (cx_all, cy_all)) = observable_data(dataset, 'centroid')

    # Retrieve images in the same order as scan_nums.
    images = [(f'scan-{sc:04d}',
            dataset[f'scan-{sc:04d}']['dvf_B1']['data'])
            for sc in scan_nums]

    fig, (ax_img, ax_cx, ax_cy) = plt.subplots(1, 3, figsize=(24, 5))

    # Left: image
    im = ax_img.imshow(images[0][1], cmap='viridis', animated=True)
    plt.colorbar(im, ax=ax_img, label='Intensity')
    ax_img.set_xlabel('Pixel X')
    ax_img.set_ylabel('Pixel Y')
    img_title = ax_img.set_title('')

    # Center: centroid X trace with a moving marker
    ax_cx.plot(xval, cx_all, marker='o',
            color='steelblue', label='Centroid X')
    marker_pt, = ax_cx.plot([], [], 'ro',
                            markersize=10, label='Current')
    ax_cx.set_xlabel(motor.capitalize())
    ax_cx.set_ylabel('Centroid X (px)')
    ax_cx.set_title('Centroid X')
    ax_cx.legend()
    ax_cx.grid(True)

    # Right: centroid Y trace with a moving marker
    ax_cy.plot(xval, cy_all, marker='o',
               color='orange', label='Centroid Y')
    marker_pt_y, = ax_cy.plot([], [], 'ro',
                              markersize=10, label='Current')
    ax_cy.set_xlabel(motor.capitalize())
    ax_cy.set_ylabel('Centroid Y (px)')
    ax_cy.set_title('Centroid Y')
    ax_cy.legend()
    ax_cy.grid(True)

    def update(frame):
        name, img = images[frame]
        im.set_data(img)
        im.set_clim(img.min(), img.max())
        img_title.set_text(f'Scan: {name}')
        marker_pt.set_data([xval[frame]], [cx_all[frame]])
        marker_pt_y.set_data([xval[frame]], [cy_all[frame]])
        return im, img_title, marker_pt, marker_pt_y

    anim = FuncAnimation(fig, update, frames=len(images), interval=300, blit=True)
    plt.tight_layout(pad=3.0)

    if save_fmt == 'gif':
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}.gif',
                writer='pillow', fps=2, dpi=300)
    elif save_fmt == 'mp4':
        # Alternatively, save as mp4 (requires ffmpeg):
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}.mp4',
                writer='ffmpeg', fps=2, dpi=300)
    else:
        print("File format not specified or unknown (use 'gif' or 'mp4' to save to file)."
              " Skipping saving process.")
        pass

    plt.close()
    display(HTML(anim.to_jshtml()))


def fwhm_plot(dataset, scanpass, wdir='.', save_fmt='gif'):
    """Plot the beam profile images and their FWHMs in a dataset.
    
    Args:
        dataset (dict): Nested dict containing the set of one full scan
            (one pass).
        scanpass (str): Identifier for the scan pass
            (used in titles and filenames).
        wdir (str): Working directory to save the plots.
            Default is current directory.
        save_fmt (str): Format to save the animation
            ('gif', 'mp4', or '' to skip saving).
    """
    # Use the existing function to get motor, positions and centroids.
    (motor, scan_nums, xval,
     (fx_all, fy_all)) = observable_data(dataset, 'fwhm')

    # Retrieve images in the same order as scan_nums.
    images = [(f'scan-{sc:04d}',
            dataset[f'scan-{sc:04d}']['dvf_B1']['data'])
            for sc in scan_nums]

    fig, (ax_img, ax_fx, ax_fy) = plt.subplots(1, 3, figsize=(24, 5))

    # Left: image
    im = ax_img.imshow(images[0][1], cmap='viridis', animated=True)
    plt.colorbar(im, ax=ax_img, label='Intensity')
    ax_img.set_xlabel('Pixel X')
    ax_img.set_ylabel('Pixel Y')
    img_title = ax_img.set_title('')

    # Center: centroid X trace with a moving marker
    ax_fx.plot(xval, fx_all, marker='o',
            color='steelblue', label='FWHM X')
    marker_pt, = ax_fx.plot([], [], 'ro',
                            markersize=10, label='Current')
    ax_fx.set_xlabel(motor.capitalize())
    ax_fx.set_ylabel('FWHM X (px)')
    ax_fx.set_title('FWHM X')
    ax_fx.legend()
    ax_fx.grid(True)

    # Right: centroid Y trace with a moving marker
    ax_fy.plot(xval, fy_all, marker='o',
               color='orange', label='FWHM Y')
    marker_pt_y, = ax_fy.plot([], [], 'ro',
                              markersize=10, label='Current')
    ax_fy.set_xlabel(motor.capitalize())
    ax_fy.set_ylabel('FWHM Y (px)')
    ax_fy.set_title('FWHM Y')
    ax_fy.legend()
    ax_fy.grid(True)

    def update(frame):
        name, img = images[frame]
        im.set_data(img)
        im.set_clim(img.min(), img.max())
        img_title.set_text(f'Scan: {name}')
        marker_pt.set_data([xval[frame]], [fx_all[frame]])
        marker_pt_y.set_data([xval[frame]], [fy_all[frame]])
        return im, img_title, marker_pt, marker_pt_y

    anim = FuncAnimation(fig, update, frames=len(images), interval=300, blit=True)
    plt.tight_layout(pad=3.0)

    if save_fmt == 'gif':
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}_fwhm.gif',
                writer='pillow', fps=2, dpi=300)
    elif save_fmt == 'mp4':
        # Alternatively, save as mp4 (requires ffmpeg):
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}_fwhm.mp4',
                writer='ffmpeg', fps=2, dpi=300)
    else:
        print("File format not specified or unknown (use 'gif' or 'mp4' to save to file)."
              " Skipping saving process.")
        pass

    plt.close()
    display(HTML(anim.to_jshtml()))





### Main entry for data reading.

In [ ]:
# workdir = "/ibira/lnls/beamlines/carcara/Carcara-X/data"
workdir = "/home/ABTLUS/arnaldo.filho/Beam_profiling/beam_profiles/2026-03-19"
file_pattern = r"mirror_tx_pass20*"


def main(workdir, file_pattern):
      files = files_in_directory(workdir, file_pattern)
      data  = data_from_h5_files(files)

      print(f"Loaded {len(files)} files matching pattern '{file_pattern}' from '{workdir}'"
            "\n Files:")
      for f in files:
            print(f" * {f}")

      return data

if __name__ == "__main__":

      # observable = 'centroid'
      # observable = 'photocollector'
      observable = 'tx'

      pass_interval = (0, 9)
      item_interval = (0, None)
      data = main(workdir, file_pattern)

### Photocollector plot for all passes.

In [ ]:
# observable = 'fwhm'
# observable = 'intensity'
# observable = 'centroid'
observable = 'photocollector'
# observable = 'z_pos'

pass_interval = (0, 9)
item_interval = (1, 12)

# Chose set interval of scans to plot.
first_pass, last_pass = pass_interval
sel_items = dict(list(data.items())[first_pass:last_pass+1])

if observable in ['intensity']:
    gr = 3
    labels = ['Peak', 'by mask', 'FWHM norm']
elif observable in ['centroid', 'fwhm']:
    gr = 2
    labels = ['X', 'Y']
else:
    gr = 1
    labels = ['X']

fr, upto = item_interval
scan_plot(sel_items, observable,
        first_item=fr, last_item=upto,
        gr=gr, labels=labels)


### Sequence of beam images.

In [ ]:
# Working directory for saving the plots.
wdir = "/home/ABTLUS/arnaldo.filho/Beam_profiling/beam_profiles"

# Plot centroids and images for all datasets.
for scanpass in data.keys():
    dataset = data[scanpass]
    centroid_plot(dataset, scanpass, wdir=wdir, save_fmt='')


#### Test with a specific scan frame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dx = 4
threshold = 10
dataset = data['03']
scandata = '0004'

img = dataset[f'scan-{scandata}']['dvf_B1']['data']
centroid = beam_centroid(dataset, 'mirror.tx')
tx, (cx, cy) = centroid[int(scandata)]

roi_avg = np.mean(img[cy-dx:cy+dx, cx-dx:cx+dx])
mean = np.mean(img)
ratio_rtom = roi_avg / mean

print(f"ROI average: {roi_avg:.2f}, Mean: {mean:.2f}, Ratio (ROI/Mean): {ratio_rtom:.2f}")

print(tx, cx, cy)
plt.imshow(img)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

img1 = data['03']['scan-0000']['dvf_A1']['data']
img2 = data['03']['scan-0000']['dvf_B1']['data']

cx = np.sum(img2, axis=0).argmax()
cy = np.sum(img2, axis=1).argmax()

ax1.imshow(img1, cmap='viridis')
ax1.set_title('dvf_A1')

ax2.imshow(img2, cmap='viridis')
ax2.set_title('dvf_B1')

print(f"Centroid (dvf_B1): {cx}, {cy}")